In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\RAG\.venv_fix\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "DATA/"
def load_pdf_files(data_path):
    loader = DirectoryLoader(data_path, glob="**/*.pdf", show_progress=True, loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [3]:

documents = load_pdf_files(DATA_PATH)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

chunks = text_splitter.split_documents(documents)
print(f"Chunk length: {len(chunks)} from {len(documents)} documents")


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [02:26<00:00, 146.62s/it]


Chunk length: 2346 from 413 documents


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(
  model_name="sentence-transformers/all-MiniLM-L6-v2",


)

text = [doc.page_content for doc in chunks]
vectors = emb.embed_documents(text)

print(f"vector length: {len(vectors)} , {len(vectors[0][:10])}")
print(vectors[0][:10])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2555.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vector length: 2346 , 10
[-0.10440604388713837, 0.016288485378026962, -0.01033782958984375, -0.0022339962888509035, -0.03919641301035881, 0.018016822636127472, -0.0569472573697567, -0.02842673845589161, -0.051072172820568085, 0.03852558135986328]


In [14]:
from pinecone import Pinecone

pc = Pinecone(
  api_key="pcsk_488GZb_GwhsowdNr38bzdioeNtRXPdfZvoLjMKyXoguyWCwNyU78YuEb1X6nXBFdL81j3z"

)


In [15]:
from pinecone import ServerlessSpec
import time

index_name = "ailawyer-384"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

while not pc.describe_index(index_name).status["ready"]:
    time.sleep(2)

index = pc.Index(index_name)


In [16]:
index

In [17]:
batch_size = 50
total_upserted = 0

for start in range(0, len(chunks), batch_size):
    batch_records = []

    for i in range(start, min(start + batch_size, len(chunks))):
        metadata = dict(chunks[i].metadata)
        metadata["text"] = chunks[i].page_content

        batch_records.append({
            "id": f"chunk-{i}",
            "values": vectors[i],
            "metadata": metadata
        })

    index.upsert(vectors=batch_records)
    total_upserted += len(batch_records)

print(f"Stored {total_upserted} chunks in Pinecone index: {index_name}")


Stored 2346 chunks in Pinecone index: ailawyer-384


In [ ]:
query = "What are Rights and Liabilities"



query_vector = emb.embed_query(query)

results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

print(results)

{'matches': [{'id': 'chunk-913',
              'metadata': {'author': 'Andrew Lyon',
                           'creationdate': '',
                           'creator': 'PyPDF',
                           'page': 204.0,
                           'page_label': '205',
                           'producer': 'Google Books PDF Converter (rel 3 '
                                       '12/12/14)',
                           'source': 'DATA\\a_guide_to_the_law_of_india.pdf',
                           'text': 'at the Presi-\n'
                                   'dency Towns to dealwith costsof '
                                   'petitionsforcertainmonies trans-\n'
                                   'ferred to Government .\n'
                                   '(SMALL CAUSE) COURT .\n'
                                   'Act 27 of1839.- An Act for authorisingthe '
                                   'Court of Requests for',
                           'title': 'A Guide to the Law of India',


In [39]:
from dotenv import load_dotenv
import os 
from langchain_groq import ChatGroq

llm = ChatGroq(model_name="llama-3.1-8b-instant",
api_key=os.getenv("GROQ_API_KEY"),
temperature=0.7,
max_tokens=500,
)

In [40]:
text_tocken = llm.invoke("What are Rights and Liabilities")
print(text_tocken)

content='**Rights and Liabilities: Understanding the Basics**\n\nRights and liabilities are two fundamental concepts that are essential in various aspects of life, including law, business, and personal relationships. Understanding the difference between rights and liabilities can help you navigate complex situations and make informed decisions.\n\n**Rights:**\n\nRights are the privileges or entitlements that an individual, group, or organization has to receive something or to act in a certain way. Rights are typically considered to be the benefits or advantages that someone has or deserves. Rights can be:\n\n1. **Legal Rights**: These are rights granted by laws, constitutions, or treaties, such as the right to free speech, the right to a fair trial, or the right to vote.\n2. **Contractual Rights**: These are rights that arise from contracts or agreements, such as the right to receive payment or the right to enforce a contract.\n3. **Social Rights**: These are rights that are recognized

In [41]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [28]:
retrieved_chunks = []

for i, match in enumerate(results["matches"], start=1):
    text = match["metadata"].get("text", "")
    score = match["score"]
    source = match["metadata"].get("source", "Unknown source")

    retrieved_chunks.append(text)

    print(f"Result {i}")
    print(f"Score: {score}")
    print(f"Source: {source}")
    print(text)
    print("-" * 80)

retrieved_chunks


Result 1
Score: 0.278727561
Source: DATA\a_guide_to_the_law_of_india.pdf
at the Presi-
dency Towns to dealwith costsof petitionsforcertainmonies trans-
ferred to Government .
(SMALL CAUSE) COURT .
Act 27 of1839.- An Act for authorisingthe Court of Requests for
--------------------------------------------------------------------------------
Result 2
Score: 0.256350517
Source: DATA\a_guide_to_the_law_of_india.pdf
placeand detainany suchPersonsinsafeCustody;
and likewisetoprovideforthe TrialofEmigrantsand theirDescen-
dantswho may exciteDisturbancesintheCountriesfrom which they
may have emigrated,and of Personsaidingthem intheProsecutionof
such attempts .
--------------------------------------------------------------------------------
Result 3
Score: 0.250038177
Source: DATA\a_guide_to_the_law_of_india.pdf
repealinglaw
intoone place, the inconvenienceof having to searchthrough
numerousvolumesforthispurposeisavoided.
Great carehasbeentaken toinsureaccuracyinthe chrono-
logicaland classifie

['at the Presi-\ndency Towns to dealwith costsof petitionsforcertainmonies trans-\nferred to Government .\n(SMALL CAUSE) COURT .\nAct 27 of1839.- An Act for authorisingthe Court of Requests for',
 'placeand detainany suchPersonsinsafeCustody;\nand likewisetoprovideforthe TrialofEmigrantsand theirDescen-\ndantswho may exciteDisturbancesintheCountriesfrom which they\nmay have emigrated,and of Personsaidingthem intheProsecutionof\nsuch attempts .',
 'repealinglaw\nintoone place, the inconvenienceof having to searchthrough\nnumerousvolumesforthispurposeisavoided.\nGreat carehasbeentaken toinsureaccuracyinthe chrono-\nlogicaland classifiedtablesaswellasintherepealingenactments.\nItisprobable,however,thatingoingover groundso extensive\nasthewholelawofIndia, some errorsmay have unavoidably\ncreptin. I shallbe extremelyobligedifthosewho make use\nof thisbook would intimateto myselforthe publishersany\nmistakesthatmay come undertheirnotice.\nA. LYON']

In [31]:
system_prompt= """ You are a knowledgeable legal assistant that provides information about laws, legal concepts, regulations, legal procedures, rights, contracts, compliance, and related legal topics based only on the retrieved legal documents.

Use the retrieved context to answer the user's question accurately, clearly, and concisely.

Guidelines:

Answer only using the information provided in the retrieved documents.
Do not make up facts, legal interpretations, statutes, regulations, or case law that are not present in the retrieved context.
If the answer is not available in the retrieved documents, respond with:
"I don't know based on the provided legal documents."
When applicable, structure your response using the following sections:
Legal Overview
Relevant Laws or Regulations
Rights and Obligations
Legal Procedures
Exceptions or Limitations
Key Considerations
Keep explanations clear, professional, and easy to understand.
Do not provide legal advice, legal representation, or definitive conclusions about a user's specific legal situation.
Do not predict court outcomes or guarantee legal results.
Mention when the retrieved information is incomplete, ambiguous, outdated, jurisdiction-specific, or uncertain.
If multiple legal interpretations are present in the retrieved documents, present them objectively and cite the relevant information from the context.
If the retrieved documents reference a specific jurisdiction, make it clear that the information applies only to that jurisdiction unless otherwise stated.

If the user asks for any topic that is not related to law or legal matters, respond with:
"I can only provide information based on the retrieved legal documents."

Retrieved Context:
{context}

Question:
{input}

Answer:
"""

In [33]:
prompt = ChatPromptTemplate.from_messages([
  ("system" , system_prompt),
  ("user", "{input}")

])

In [34]:
doc_chain=create_stuff_documents_chain(llm,prompt)
doc_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template=' You are a knowledgeable legal assistant that provides information about laws, legal concepts, regulations, legal procedures, rights, contracts, compliance, and related legal topics based only on the retrieved legal documents.\n\nUse the retrieved context to answer the user\'s question accurately, clearly, and concisely.\n\nGuidelines:\n\nAnswer only using the information provided in the retrieved documents.\nDo not make up facts, legal interpretations, statutes, regulations, or case law that are not present in the retrieved context.\nIf the answer is not available in t

In [43]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain_lcel = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: RunnableLambda(pinecone_retriever)
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template=' You are a knowledgeable legal assistant that provides information about laws, legal concepts, regulations, legal procedures, rights, contracts, compliance, and related legal topics based only on the retrieved legal documents.\n\nUse the retrieved context to answer the user\'s question accurately, clearly, and concisely.\n\nGuidelines:\n\nAnswer only using the information provided in the retrieved documents.\nDo not make up facts, legal interpretations, statutes, regulations, or case law that are not present in the retrieved context.\nIf the answer is not available in the retrieved documents, respond with:\n"I don\'t know base

In [45]:
def query_rag_lcel(input):
  print("Question:{input}")
  print("**************")

  answer = rag_chain_lcel.invoke(input)
  print("Answer:{answer}")
  return answer

query_rag_lcel("What are Rights and Liabilities")

Question:{input}
**************
Answer:{answer}


"**Legal Overview**\n\nRights and liabilities refer to the claims or entitlements of an individual or entity, and the corresponding obligations or responsibilities that arise from these claims.\n\n**Relevant Laws or Regulations**\n\nThe concept of rights and liabilities is not specifically defined in the retrieved documents, but it is mentioned as the result of laws that are conferred through some event or action.\n\n**Rights and Obligations**\n\nAccording to the retrieved documents, rights are typically enforced through action in law courts, while equitable rights are enforced through a suit in chancery courts. Rights and liabilities are the results of laws, but they do not arise directly from the law. They are conferred through some factor or event, which can include marriage.\n\nFor example, upon marriage, a husband has a right to the property of his wife or to the management of it, and a right to compel any person who detains her to give her up. The wife has a right to maintenance 